In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd

#1番最初のセルで実行して「ストック」しておく
def run_lgbm_cv_proba(X, y):
    """
    ヒント(X)と答え(y)を渡すだけで、自動で型変換から
    5分割クロスバリデーション（確率予測）までを行う関数
    """
    # 文字列の列を自動で一括 category 型に
    X_copy = X.copy() # 元のデータを壊さないための安全マージン
    obj_cols = X_copy.select_dtypes(include=['object']).columns
    X_copy[obj_cols] = X_copy[obj_cols].astype('category')
    
    # 2. 5分割のクロスバリデーションの準備
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds_lgb_proba = np.zeros(len(X_copy))
    
    print("--- 🤖 LightGBM クロスバリデーション（確率版）開始 ---")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_copy, y)):
        X_train, y_train = X_copy.iloc[train_idx], y[train_idx]
        X_val, y_val = X_copy.iloc[val_idx], y[val_idx]
        
        model = LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05)
        model.fit(X_train, y_train)
        
        # 確率（1である確率）を予測して箱に保存
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        oof_preds_lgb_proba[val_idx] = y_pred_proba
        
        # 各Foldの手元スコア確認
        score = accuracy_score(y[val_idx], np.where(y_pred_proba > 0.5, 1, 0))
        print(f"Fold {fold+1} の正解率: {score:.5f}")
        
    # 🌟【重要！】全体のガチの実力スコアを表示
    total_score = accuracy_score(y, np.where(oof_preds_lgb_proba > 0.5, 1, 0))
    print(f"✨ 全体（OOF）の最終CVスコア: {total_score:.5f}")
        
    # 🌟【ここを追加！】計算した確率の配列を、関数の外に呼び出し元へ戻す
    return oof_preds_lgb_proba

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

# 👇 これをストック用として1番最初のセルで実行しておく
def run_xgboost_cv_proba(X, y):
    """
    ヒント(X)と答え(y)を渡すだけで、XGBoost特有の型変換を行い、
    5分割クロスバリデーション（確率予測）までを行う関数
    """
    X_copy = X.copy() # 元のデータを壊さないための安全マージン
    
    # 1. XGBoost用に、一回 str にしてから category 型に変換（インデント修正）
    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        X_copy[col] = X_copy[col].astype(str).astype('category')
        
    # 2. 5分割のクロスバリデーションの準備（関数内で定義）
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds_xgb_proba = np.zeros(len(X_copy))
    
    print("--- 🔥 XGBoost クロスバリデーション（確率版）開始 ---")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_copy, y)):
        X_train, y_train = X_copy.iloc[train_idx], y[train_idx]
        X_val, y_val = X_copy.iloc[val_idx], y[val_idx]
        
        # enable_categorical=True を忘れずセット
        model_xgb = XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.05, enable_categorical=True)
        model_xgb.fit(X_train, y_train)
        
        # 確率で予測を出して箱に保存
        y_pred_proba_xgb = model_xgb.predict_proba(X_val)[:, 1]
        oof_preds_xgb_proba[val_idx] = y_pred_proba_xgb
        
    # 3. 全部のFoldが終わった後に、全体のスコアを計算
    xgb_actual_score = accuracy_score(y, np.where(oof_preds_xgb_proba > 0.5, 1, 0))
    print(f"XGBoostのCVスコア: {xgb_actual_score:.5f}")
    
    # 🌟 計算した確率の配列を外に戻す
    return oof_preds_xgb_proba

In [ ]:
# 1行目：LightGBMの確率をゲット
lgb_proba = run_lgbm_cv_proba(X, y)

# 2行目：XGBoostの確率をゲット
xgb_proba = run_xgboost_cv_proba(X, y)

# 3行目：2つを足して2で割って、最強のアンサンブルCVスコアを確認！
ensemble_score = accuracy_score(y, np.where((lgb_proba + xgb_proba) / 2 > 0.5, 1, 0))
print(f"2大AI融合の最終CVスコア: {ensemble_score:.5f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 👇 本日の主役：説明性重視の分類一括オーディション関数
def run_classification_audition(X, y, test_size=0.2, random_state=42):
    """
    ヒント(X)と答え(y)を渡すだけで、標準化前処理から
    「ロジスティック回帰」「決定木」「ランダムフォレスト」を一括評価する関数
    """
    # 1. データの分割
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    # 2. パイプラインの定義（すべてscikit-learnの、説明しやすいモデルたち）
    pipelines = {
        'LogisticRegression（ロジスティック回帰）': Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(random_state=random_state))
        ]),
        'DecisionTree（決定木）': Pipeline([
            ('scaler', StandardScaler()),
            ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state)) # 見やすさ重視で深さを3に制限
        ]),
        'RandomForest（ランダムフォレスト）': Pipeline([
            ('scaler', StandardScaler()),
            ('rf', RandomForestClassifier(random_state=random_state))
        ])
    }
    
    print("--- ⚙️ scikit-learn 分類一括オーディション開始 ---")
    results = {}
    for name, pipe in pipelines.items():
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        
        # 今回は2択の「正解率（Accuracy）」で競います
        score = accuracy_score(y_val, y_pred)
        print(f"{name} の正解率: {score:.5f}")
        results[name] = pipe
        
    return results

In [ ]:
# 1. 3つのモデルの結果が入った箱（results）から、それぞれのAIを取り出す
for name, pipe in results.items():
    print(f"\n---  {name} の重要度を調査 ---")
    
    # 決定木やランダムフォレストは 'dt' や 'rf' という名前でパイプラインに入れています
    # モデルの本体を取り出すための辞書を用意
    model_name = list(pipe.named_steps.keys())[1] # パイプラインの2番目（[0]=scaler, [1]=model）
    model = pipe.named_steps[model_name]
    
    # 【ロジスティック回帰の場合】（係数を取得）
    if name == 'LogisticRegression':
        importances = model.coef_[0]
        
    # 【決定木・ランダムフォレストの場合】（重要度を取得）
    else:
        importances = model.feature_importances_
        
    # 表にまとめてグラフ化
    df_imp = pd.DataFrame({'Feature': X.columns, 'Importance': abs(importances)}).sort_values(by='Importance', ascending=False)
    
    # グラフ表示
    plt.figure(figsize=(8, 4))
    sns.barplot(x='Importance', y='Feature', data=df_imp.head(10), palette='magma')
    plt.title(f'Top 10 Features: {name}')
    plt.show()